In [ ]:
%load_ext autoreload
%autoreload 2

### Match stations, using station name, lat, lon. The name matching in Raw data meta_data form, in the folder and in the database

In [2]:
from crmprtd import Row
import logging
import os
import pickle
import pandas as pd
import numpy as np
from natsort import natsorted
from natsort import natsort_keygen
from pprint import pprint
from collections import Counter
from collections import defaultdict

import re
from rapidfuzz import fuzz
from crmprtd import infer
from fern_func import *
from sql_func import *
import sqlalchemy as sa
from fern_raw_db_dompare import *

In [3]:
# --- Main workflow ---
# --- read data ---
meta_path = '/fern_data/FERNNorth2024_VF/20241125 MeteorologicalNetworks-FERN-VF-shared.xlsx'

df_station = pd.read_excel(meta_path)
df_station = df_station.sort_values(by='station_name', key=natsort_keygen())


# --- output folder ---
output_folder = '/workspaces/crmprtd/fern/all_stations_insert/rows_output/'
os.makedirs(output_folder, exist_ok=True)


### Summarize the fern data raw data into a CSV

In [19]:
engine = sa.create_engine("postgresql+psycopg2://tongli1997@/crmp?host=pg01.pcic.uvic.ca,pg02.pcic.uvic.ca&port=5432,5432&target_session_attrs=read-write&passfile=/workspaces/crmprtd/.pgpass", echo=False)
session = Session(engine)


# --- Manual corrections dictionary ---
manual_corrections = {
    'Atlin School': 'Atlin',
    'BowronPit': 'Bowron Pit',
    'ChiefLakeWx': 'Chief Lake',
    'MacJxnWx': 'Mackenzie Jxn',
    'BoulderWx': 'Boulder Creek',
    'Dennis': 'Dennis',
    'DunsterWx': 'DUNSTER',
    'Tamarac':'BednestiWx'
}

# -------------------------------
# Loop over all stations
summary_fern_raw_records = []
summary_db_records = []
summary_fern_db_exist_records = []
no_match_station_name = []

data_path = '/fern_data/FERNNorth2024_VF/WxData24'
csv_files = [f for f in os.listdir(data_path) if f.endswith('.csv')]
# Sort in natural order
csv_files = natsorted(csv_files)

for station_i, csv_file in enumerate(csv_files):
    station_row = df_station.iloc[station_i]
    station_name = station_row[ 'station_name']

    #### summarize raw data
    records = summarize_raw_station(csv_file, station_row, data_path)
    summary_fern_raw_records.extend(records)  # add to the master list

    #### summarize db data
    df_db = get_station_vars_time_from_db(engine, station_name, manual_corrections)

    if isinstance(df_db, tuple):
        # in case your function still returns a tuple
        df_db = df_db[0]

    summary_db_records.append(df_db)


# Create summary DataFrame
raw_summary = pd.DataFrame(summary_fern_raw_records)
db_summary = pd.concat(summary_db_records, ignore_index=True)

# Save to CSV if needed
raw_summary_path = os.path.join(output_folder, "Fern_raw_station_variable_summary.csv")
raw_summary.to_csv(raw_summary_path, index=False)
print(f"✅ Summary saved to {raw_summary_path}")

db_summary_path = os.path.join(output_folder, "Fern_db_station_variable_summary.csv")
db_summary.to_csv(db_summary_path, index=False)

# View summary
raw_summary.head()
db_summary.head()

📍 Station: Atlin School (File: Atlin school)
❌ No DB match for 'Atlin School'
📍 Station: BarrenWx (File: Barren)
⚠️ Variable 'Wetness' is all NaN, skipping
⚠️ Variable 'Snow depth raw' is all NaN, skipping
⚠️ Variable 'Snow depth' is all NaN, skipping
⚠️ Variable 'Temp 2' is all NaN, skipping
⚠️ Variable 'RH 2' is all NaN, skipping
⚠️ Variable 'DewPt 2' is all NaN, skipping
✅ 'BarrenWx' matched DB as 'BarrenWx' (LIKE) → 13 vars found.
📍 Station: BlackhawkWx (File: Blackhawk)
⚠️ Variable 'Wetness' is all NaN, skipping
⚠️ Variable 'Water Content' is all NaN, skipping
⚠️ Variable 'Water Content' is all NaN, skipping
⚠️ Variable 'Temp 2' is all NaN, skipping
⚠️ Variable 'RH 2' is all NaN, skipping
⚠️ Variable 'DewPt 2' is all NaN, skipping
✅ 'BlackhawkWx' matched DB as 'Blackhawk' (LIKE) → 11 vars found.
📍 Station: BoulderWx (File: BoulderCr)
⚠️ Variable 'Snow depth raw' is all NaN, skipping
⚠️ Variable 'Snow depth' is all NaN, skipping
⚠️ Variable 'Water Content' is all NaN, skipping
⚠️ V

/tmp/ipykernel_2042/2222454989.py:49: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  db_summary = pd.concat(summary_db_records, ignore_index=True)


,station_name,matched_name,lat,lon,net_var_name,unit,earliest_time,latest_time
0,BarrenWx,BarrenWx,54.51044,-126.6143,DewPtC,celsius,2021-07-09 12:00:00,2024-10-03 11:00:00
1,BarrenWx,BarrenWx,54.51044,-126.6143,GustSpeedms,m s-1,2021-07-09 12:00:00,2024-10-03 11:00:00
2,BarrenWx,BarrenWx,54.51044,-126.6143,Pressurembar,millibar,2021-07-09 12:00:00,2024-10-03 11:00:00
3,BarrenWx,BarrenWx,54.51044,-126.6143,Rainmm,mm,2021-07-09 13:00:00,2024-10-03 11:00:00
4,BarrenWx,BarrenWx,54.51044,-126.6143,RH,%,2021-07-09 12:00:00,2024-10-03 11:00:00


### Compare the station, according to the lat, lon

In [20]:
# --- Compare all stations ---
threshold_km = 10  # tolerance for "same" station
results = []

for station_name in df_station["station_name"]:
    raw_one = raw_summary.loc[raw_summary["station_name"] == station_name]
    db_one = db_summary.loc[db_summary["station_name"] == station_name]

    raw_lat = raw_one["lat"].unique()[0]
    raw_lon = raw_one["lon"].unique()[0]

    # If station not found in DB
    if db_one.empty:
        results.append({
            "station_name": station_name,
            "db_matched_name":np.nan,
            "raw_lat": raw_lat,
            "raw_lon": raw_lon,
            "db_lat": np.nan,
            "db_lon": np.nan,
            "distance_km": np.nan,
            "match": "❌ not in DB"
        })
        print(f"\n🛰️ {station_name}")
        print(f"  Raw lat/lon: {raw_lat:.4f}, {raw_lon:.4f}")
        print("  ❌ No match found in database")
        continue

    # Extract DB coordinates
    db_lat = db_one["lat"].unique()[0]
    db_lon = db_one["lon"].unique()[0]

    # Compare
    same, dist = same_station(raw_lat, raw_lon, db_lat, db_lon, threshold_km)
    db_matched_name = db_one["matched_name"].unique()[0]

    results.append({
        "station_name": station_name,
        "db_matched_name":db_matched_name,
        "raw_lat": raw_lat,
        "raw_lon": raw_lon,
        "db_lat": db_lat,
        "db_lon": db_lon,
        "distance_km": dist,
        "match": "✅ same" if same else "⚠️ different"
    })

    print(f"\n🛰️ {station_name}")
    print(f"  Raw lat/lon: {raw_lat:.4f}, {raw_lon:.4f}")
    print(f"  DB  lat/lon: {db_lat:.4f}, {db_lon:.4f}")
    print(f"  Distance: {dist:.3f} km → {'✅ same' if same else '⚠️ different'}")

# --- Create summary DataFrame ---
df_compare = pd.DataFrame(results)

# Optional: save
df_compare.to_csv(os.path.join(output_folder, "station_location_compare.csv"), index=False)


🛰️ Atlin School
  Raw lat/lon: 59.5740, -133.6870
  ❌ No match found in database

🛰️ BarrenWx
  Raw lat/lon: 54.5104, -126.6143
  DB  lat/lon: 54.5104, -126.6143
  Distance: 0.003 km → ✅ same

🛰️ BlackhawkWx
  Raw lat/lon: 55.0789, -120.3522
  DB  lat/lon: 55.0789, -120.3520
  Distance: 0.011 km → ✅ same

🛰️ BoulderWx
  Raw lat/lon: 55.1082, -127.3747
  DB  lat/lon: 55.1082, -127.3750
  Distance: 0.017 km → ✅ same

🛰️ BowronPit
  Raw lat/lon: 53.9021, -122.0154
  DB  lat/lon: 53.9021, -122.0154
  Distance: 0.000 km → ✅ same

🛰️ BulkleyWx
  Raw lat/lon: 53.7722, -122.7207
  DB  lat/lon: 53.7720, -122.7209
  Distance: 0.029 km → ✅ same

🛰️ CPFWx
  Raw lat/lon: 53.7538, -122.7189
  DB  lat/lon: 53.7538, -122.7188
  Distance: 0.005 km → ✅ same

🛰️ Canoe Mountain Stn
  Raw lat/lon: 52.7098, -119.1913
  DB  lat/lon: 52.7098, -119.1910
  Distance: 0.018 km → ✅ same

🛰️ Caribou Pass
  Raw lat/lon: 58.3572, -129.5464
  DB  lat/lon: 58.3572, -129.5464
  Distance: 0.000 km → ✅ same

🛰️ ChapmanWx

#### For the stations are not in the database, using lat, lon to search for data in the database

In [21]:
not_in_db = df_compare[df_compare["match"] == "❌ not in DB"]
results = []

for _, row in not_in_db.iterrows():
    station_name = row["station_name"]
    lat, lon = row["raw_lat"], row["raw_lon"]
    print(f"\n🔍 Searching nearby stations for {station_name} ({lat}, {lon})...")

    nearby = find_nearby_stations(engine, lat, lon, radius_km=10)
    
    if nearby.empty:
        print("  ❌ No nearby stations found within 10 km")
        results.append({
            "station_name": station_name,
            "raw_lat": lat,
            "raw_lon": lon,
            "nearest_db_station": None,
            "distance_m": None
        })
    else:
        nearest = nearby.iloc[0]  # closest station
        print(f"  ✅ Nearest station: {nearest['station_name']}, distance {nearest['distance_m']:.1f} m")
        results.append({
            "station_name": station_name,
            "raw_lat": lat,
            "raw_lon": lon,
            "nearest_db_station": nearest["station_name"],
            "distance_m": nearest["distance_m"]
        })

# Convert results to a DataFrame
df_nearby = pd.DataFrame(results)

df_nearby["distance_km"] = df_nearby["distance_m"] / 1000
df_nearby_subset = df_nearby[["station_name", "nearest_db_station", "distance_km"]]
df_nearby_subset = df_nearby_subset.rename(columns={"distance_km": "nearby_distance_km"})
# Merge
df_compare_updated = df_compare.merge(
    df_nearby_subset,
    on="station_name",
    how="left"
)
df_compare_updated.to_csv(os.path.join(output_folder, "station_location_compare.csv"), index=False)


🔍 Searching nearby stations for Atlin School (59.574, -133.687)...
  ❌ No nearby stations found within 10 km

🔍 Searching nearby stations for CrookedLk (59.88626, -126.42886)...
  ❌ No nearby stations found within 10 km

🔍 Searching nearby stations for IBB2Wx (54.81127, -126.92067)...
  ❌ No nearby stations found within 10 km

🔍 Searching nearby stations for IBB3Wx (54.71172, -127.30577)...
  ✅ Nearest station: Hudson Bay Mtn2, distance 7987.5 m

🔍 Searching nearby stations for Inklin (58.900859, -133.140416)...
  ❌ No nearby stations found within 10 km

🔍 Searching nearby stations for SeebachWx (54.355, 122.058)...
  ❌ No nearby stations found within 10 km


## match the variable

In [22]:
raw_summary_path = os.path.join(output_folder, "Fern_raw_station_variable_summary.csv")
db_summary_path = os.path.join(output_folder, "Fern_db_station_variable_summary.csv")

raw_summary = pd.read_csv(raw_summary_path)
db_summary = pd.read_csv(db_summary_path)

all_matches = []

for station_name in df_station["station_name"]:
    raw_one = raw_summary.loc[raw_summary["station_name"] == station_name]
    db_one = db_summary.loc[db_summary["station_name"] == station_name]

    station_matches = fuzzy_match_station_vars(raw_one, db_one, threshold=80)
    all_matches.append(station_matches)

# Combine all station matches
# Filter out empty DataFrames before concatenation
non_empty_matches = [df for df in all_matches if not df.empty]

if non_empty_matches:
    matches_df = pd.concat(non_empty_matches, ignore_index=True)
else:
    matches_df = pd.DataFrame(columns=["station_name", "variable", "db_var_match", "score"])
    
# Merge with raw_summary
raw_summary_matched = raw_summary.merge(matches_df, on=["station_name", "variable"], how="left")


In [23]:
#### merging the information in the database into the same match form
raw_summary_enriched = raw_summary_matched.merge(
    db_summary[['station_name','matched_name', 'net_var_name', 'unit', 'earliest_time', 'latest_time']],
    left_on=['station_name', 'db_var_match'],   # match station + variable
    right_on=['station_name', 'net_var_name'],
    how='left'
)
raw_summary_enriched = raw_summary_enriched.drop(columns=['net_var_name'])

raw_summary_enriched = raw_summary_enriched.rename(
    columns={
        "unit_x": "unit_raw",
        "earliest_time_x": "earliest_time_raw",
        "latest_time_x": "latest_time_raw",
        "unit_y": "unit_db",
        "earliest_time_y": "earliest_time_db",
        "latest_time_y": "latest_time_db",
        "score": "variable_match_score",

    }
)

# Convert to datetime if they’re strings
for col in ["earliest_time_raw", "latest_time_raw", "earliest_time_db", "latest_time_db"]:
    raw_summary_enriched[col] = pd.to_datetime(raw_summary_enriched[col], errors="coerce")

# Calculate time gaps (in days)
raw_summary_enriched["earliest_diff_days"] = (
    (raw_summary_enriched["earliest_time_raw"] - raw_summary_enriched["earliest_time_db"]).dt.days
)

raw_summary_enriched["latest_diff_days"] = (
    (raw_summary_enriched["latest_time_raw"] - raw_summary_enriched["latest_time_db"]).dt.days
)
raw_summary_enriched = raw_summary_enriched.drop_duplicates()
# Save
output_path = os.path.join(output_folder, "Fern_raw_db_station_matched_v1.csv")
raw_summary_enriched.to_csv(output_path, index=False)

print(f"✅ Done. Matches saved to {output_path}")

✅ Done. Matches saved to /workspaces/crmprtd/fern/all_stations_insert/rows_output/Fern_raw_db_station_matched_v1.csv


### Using the information in the form, generate Rows for those data

In [24]:
data_insert_batch1 = raw_summary_enriched[
    (raw_summary_enriched["earliest_diff_days"].abs() > 0) |
    (raw_summary_enriched["latest_diff_days"].abs() > 0)
]
data_insert_batch1

,station_name,native_id,station_file_name,lat,lon,elev,variable,unit_raw,earliest_time_raw,latest_time_raw,db_var_match,variable_match_score,matched_name,unit_db,earliest_time_db,latest_time_db,earliest_diff_days,latest_diff_days
13,BarrenWx,Barren,Barren,54.510444,-126.614341,1051,Rain,mm,2021-07-09 12:00:00,2024-10-03 11:00:00,Rainmm,80.000000,BarrenWx,mm,2021-07-09 13:00:00,2024-10-03 11:00:00,-1.0,0.0
33,BlackhawkWx,Blackhawk,Blackhawk,55.078885,-120.352171,962,Rain,mm,2012-05-24 13:00:00,2025-01-07 10:00:00,Rainmm,80.000000,Blackhawk,mm,2012-05-24 14:00:00,2025-01-07 10:00:00,-1.0,0.0
43,BlackhawkWx,Blackhawk,Blackhawk,55.078885,-120.352171,962,Snow depth,cm,2012-05-24 13:00:00,2025-01-07 10:00:00,Snow_Depth,80.000000,Blackhawk,cm,2016-10-26 14:00:00,2025-01-07 10:00:00,-1617.0,0.0
47,BoulderWx,Boulder Creek,BoulderCr,55.108173,-127.374740,385,Rain,mm,2010-06-07 13:00:00,2024-08-07 08:00:00,Rainmm,80.000000,Boulder Creek,mm,2010-06-07 13:00:00,2023-09-21 14:00:00,0.0,320.0
48,BoulderWx,Boulder Creek,BoulderCr,55.108173,-127.374740,385,Pressure,mbar,2010-06-07 13:00:00,2024-08-07 08:00:00,Pressurembar,80.000000,Boulder Creek,millibar,2010-06-07 13:00:00,2023-09-21 14:00:00,0.0,320.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
454,Willow-BowronWx,PGTIS2,WillowBowron PGTIS 2,53.772577,-122.724634,596,Wind Speed,m/s,2007-07-31 15:00:00,2024-10-24 09:00:00,WindSpeedms,85.714286,Willow-BowronWx,m s-1,2007-07-31 15:00:00,2023-10-17 09:00:00,0.0,373.0
455,Willow-BowronWx,PGTIS2,WillowBowron PGTIS 2,53.772577,-122.724634,596,Gust Speed,m/s,2007-07-31 15:00:00,2024-10-24 09:00:00,GustSpeedms,85.714286,Willow-BowronWx,m s-1,2007-07-31 15:00:00,2023-10-17 09:00:00,0.0,373.0
456,Willow-BowronWx,PGTIS2,WillowBowron PGTIS 2,53.772577,-122.724634,596,Wind Direction,ø,2007-07-31 15:00:00,2024-10-24 09:00:00,WindDirection,96.296296,Willow-BowronWx,degree,2007-07-31 15:00:00,2023-10-17 09:00:00,0.0,373.0
457,Willow-BowronWx,PGTIS2,WillowBowron PGTIS 2,53.772577,-122.724634,596,Solar Radiation,W/m²,2007-07-31 15:00:00,2024-10-24 09:00:00,SolarRadiationWm,90.322581,Willow-BowronWx,W m-2,2007-07-31 15:00:00,2023-10-17 09:00:00,0.0,373.0


In [26]:

def extract_new_data_rows(row, data_path, network_name="FLNRO-FERN"):
    """
    Extract time-value pairs outside the database range for a given station variable.

    Parameters
    ----------
    row : pd.Series
        A row from `data_insert_batch1` containing station/variable info.
    data_path : str
        Directory path containing the raw station CSV files.
    network_name : str, optional
        Network name for Row construction (default: 'FLNRO-FERN').

    Returns
    -------
    list[Row]
        A list of Row objects for data outside the database time range.
    """
    # --- Extract info from the row ---
    variable_name_db = row['db_var_match']
    unit_db = row['unit_db']
    native_id = row['native_id']
    lat = float(row['lat'])
    lon = float(row['lon'])

    # --- Read the CSV file ---
    csv_file_name = row['station_file_name'] + '.csv'
    csv_path = os.path.join(data_path, csv_file_name)
    df_data = pd.read_csv(csv_path, encoding='latin1', low_memory=False)

    # --- Determine the time column ---
    time_col = 'Date' if 'Date' in df_data.columns else df_data.columns[0]
    df_data[time_col] = pd.to_datetime(df_data[time_col], errors='coerce')

    # --- Construct variable column name (e.g. "Rain, mm") ---
    variable_name = row['variable'] + ', ' + row['unit_raw']

    # --- Define DB time range ---
    earliest_db = pd.to_datetime(row['earliest_time_db'])
    latest_db = pd.to_datetime(row['latest_time_db'])

    # --- Select rows outside the DB time range ---
    df_outside = df_data[
        (df_data[time_col] < earliest_db) | (df_data[time_col] > latest_db)
    ][[time_col, variable_name]].dropna(subset=[variable_name]).reset_index(drop=True)

    # --- Build list of Row objects ---
    rows = [
        Row(
            time=row_i[time_col],
            val=row_i[variable_name],
            variable_name=variable_name_db,
            unit=unit_db,
            network_name=network_name,
            station_id=native_id,
            lat=lat,
            lon=lon
        )
        for _, row_i in df_outside.iterrows()
    ]

    return rows



In [27]:
data_path = '/fern_data/FERNNorth2024_VF/WxData24'

row1 = data_insert_batch1.iloc[0]
rows_to_insert = extract_new_data_rows(row1, data_path)

print(f"{len(rows_to_insert)} new records found for {row1['station_name']}.")

0 new records found for BarrenWx.


In [28]:
all_rows = []

for i, row in data_insert_batch1.iterrows():
    station = row["station_name"]
    variable = row["variable"]

    print(f"\n🔍 [{i+1}/{len(data_insert_batch1)}] Processing {station} — {variable}")

    new_rows = extract_new_data_rows(row, data_path)
    all_rows.extend(new_rows)

    if len(new_rows) > 0:
        print(f"  ✅ {len(new_rows)} new records found outside DB time range.")
    else:
        print("  ⚠️ No new records found (matches DB time range exactly).")


🔍 [14/231] Processing BarrenWx — Rain
  ⚠️ No new records found (matches DB time range exactly).

🔍 [34/231] Processing BlackhawkWx — Rain
  ⚠️ No new records found (matches DB time range exactly).

🔍 [44/231] Processing BlackhawkWx — Snow depth
  ⚠️ No new records found (matches DB time range exactly).

🔍 [48/231] Processing BoulderWx — Rain
  ✅ 7696 new records found outside DB time range.

🔍 [49/231] Processing BoulderWx — Pressure
  ✅ 7697 new records found outside DB time range.

🔍 [50/231] Processing BoulderWx — Temp
  ✅ 7697 new records found outside DB time range.

🔍 [51/231] Processing BoulderWx — RH
  ✅ 7697 new records found outside DB time range.

🔍 [52/231] Processing BoulderWx — DewPt
  ✅ 7697 new records found outside DB time range.

🔍 [53/231] Processing BoulderWx — Wind Speed
  ✅ 7697 new records found outside DB time range.

🔍 [54/231] Processing BoulderWx — Gust Speed
  ✅ 7697 new records found outside DB time range.

🔍 [55/231] Processing BoulderWx — Wind Direction

## Insert code

In [29]:
from auto_insert_func import *

rows_insert = all_rows
db_url = "postgresql+psycopg2://tongli1997@/crmp?host=pg01.pcic.uvic.ca,pg02.pcic.uvic.ca&port=5432,5432&target_session_attrs=read-write&passfile=/workspaces/crmprtd/.pgpass"

log_file_path = '/workspaces/crmprtd/fern/all_stations_insert/fern_all_station_log.log'

insert_rows(rows_insert, log_file_path, db_url)


{"asctime": "2025-11-12 18:43:52,815", "levelname": "DEBUG", "name": "crmprtd.align", "message": "Searching for native_id = ChiefWx"}
{"asctime": "2025-11-12 18:43:52,906", "levelname": "DEBUG", "name": "crmprtd.align", "message": "Found exactly one matching history_id"}
{"asctime": "2025-11-12 18:43:52,936", "levelname": "DEBUG", "name": "crmprtd.align", "message": "Cache miss: find_or_create_matching_history_and_station ('FLNRO-FERN', 'ChiefWx', 54.238028, -122.877306, ('diagnostic', False)) -> <pycds.orm.tables.History object at 0xfffe9d639d60>"}
{"asctime": "2025-11-12 18:43:52,937", "levelname": "DEBUG", "name": "crmprtd.align", "message": "Searching for native_id = Bednesti"}
{"asctime": "2025-11-12 18:43:52,942", "levelname": "DEBUG", "name": "crmprtd.align", "message": "Found exactly one matching history_id"}
{"asctime": "2025-11-12 18:43:52,944", "levelname": "DEBUG", "name": "crmprtd.align", "message": "Cache miss: find_or_create_matching_history_and_station ('FLNRO-FERN', 'B

: 